# 01 · EDA de órdenes (Bronze → Gold)

Notebook semilla del **Data Science & MLOps Agent** (ver `AGENTS.md` §2.5).

- Fuente analítica: `marts.fct_orders` / `marts.dim_customer` en ClickHouse (`clickhouse:8123`).
- Fuente cruda (opcional): Parquet en `s3://datalake/bronze/orders/dt=*` (MinIO).
- Nunca se lee el OLTP ni Bronze desde un dashboard: este notebook es para exploración, y todo
  hallazgo reutilizable debe terminar como modelo dbt o como *feature* en `marts.customer_features`.

In [ ]:
import os
import pandas as pd
import clickhouse_connect

CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "clickhouse")
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "dbt")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "dbt_dev_pwd")

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST, port=8123,
    username=CLICKHOUSE_USER, password=CLICKHOUSE_PASSWORD,
    database="marts",
)
client.command("SELECT version()")

In [ ]:
# Serie diaria de ingresos y margen (últimos 90 días)
sql = """
SELECT d.date_day AS dia,
       count()                      AS pedidos,
       sum(f.total_amount)          AS ingresos,
       sum(f.gross_margin)          AS margen
FROM   marts.fct_orders f
JOIN   marts.dim_date   d ON d.date_key = f.date_key
WHERE  d.date_day >= today() - 90
GROUP BY dia
ORDER BY dia
"""
serie = client.query_df(sql)
serie.tail(10)

In [ ]:
# Distribución de la etiqueta de churn usada por el modelo churn_classifier
features = client.query_df("SELECT * FROM marts.customer_features")
features["churn_label"].value_counts(normalize=True).rename("tasa").to_frame()

## Siguientes pasos del agente

1. Validar supuestos con `governance` (contratos y sesgo) antes de entrenar.
2. Entrenar con `python /opt/airflow/pipelines/train_churn.py --run-id <run_id>`.
3. Registrar métricas y firma en MLflow (`http://localhost:5000`).
4. Propagar el `run_id` al dashboard para mantener la trazabilidad extremo a extremo.